# EPC Data Quality Audit: Camden, London
## 1. Data profiling

Source: Domestic Energy Performance Certificates, Camden, Jan 2012 - Jun 2026,
downloaded from get-energy-performance-data.communities.gov.uk (data not included
in repo due to licence).

In [22]:
import pandas as pd

df = pd.read_csv("../data/raw/camden_certificates.csv", low_memory=False)
print(df.shape)

(102185, 93)


In [23]:
df.head()

,certificate_number,address1,address2,address3,postcode,posttown,address,constituency,constituency_label,local_authority,...,walls_env_eff,wind_turbine_count,windows_description,windows_energy_eff,windows_env_eff,floor_env_eff,region,country,uprn,uprn_source
0,9637-2800-7390-9224-9051,"3, Agar Grove",NaN,NaN,NW1 9SL,LONDON,"3, Agar Grove",E14001290,Holborn and St Pancras,E09000007,...,Very Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,5055943.0,Energy Assessor
1,0060-2855-6531-9721-2181,"18, Brassey Road",NaN,NaN,NW6 2BE,LONDON,"18, Brassey Road",E14001265,Hampstead and Highgate,E09000007,...,Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,5003408.0,Energy Assessor
2,0125-3852-7281-9794-8445,7 Pickstock Court,155 Gray's Inn Road,NaN,WC1X 8UE,LONDON,"7 Pickstock Court, 155 Gray's Inn Road",E14001290,Holborn and St Pancras,E09000007,...,Very Good,0.0,High performance glazing,Very Good,Very Good,NaN,E12000007,England,5172370.0,Energy Assessor
3,0248-4906-7239-2994-6924,14b Agamemnon Road,NaN,NaN,NW6 1DY,LONDON,14b Agamemnon Road,E14001265,Hampstead and Highgate,E09000007,...,Very Poor,0.0,Single glazed,Very Poor,Very Poor,NaN,E12000007,England,NaN,NaN
4,0296-2873-6747-9592-3255,Rear Garden Flat,"61, Belsize Park Gardens",NaN,NW3 4JN,LONDON,"Rear Garden Flat, 61, Belsize Park Gardens",E14001265,Hampstead and Highgate,E09000007,...,Very Poor,0.0,Partial double glazing,Poor,Poor,NaN,E12000007,England,5115246.0,Energy Assessor


In [24]:
df.columns.tolist()

['certificate_number',
 'address1',
 'address2',
 'address3',
 'postcode',
 'posttown',
 'address',
 'constituency',
 'constituency_label',
 'local_authority',
 'local_authority_label',
 'built_form',
 'co2_emiss_curr_per_floor_area',
 'co2_emissions_current',
 'co2_emissions_potential',
 'construction_age_band',
 'current_energy_efficiency',
 'current_energy_rating',
 'energy_consumption_current',
 'energy_consumption_potential',
 'energy_tariff',
 'environment_impact_current',
 'environment_impact_potential',
 'extension_count',
 'fixed_lighting_outlets_count',
 'flat_storey_count',
 'flat_top_storey',
 'floor_description',
 'floor_energy_eff',
 'floor_height',
 'floor_level',
 'glazed_area',
 'glazed_type',
 'heat_loss_corridor',
 'heating_cost_current',
 'heating_cost_potential',
 'hot_water_cost_current',
 'hot_water_cost_potential',
 'hot_water_energy_eff',
 'hot_water_env_eff',
 'hotwater_description',
 'inspection_date',
 'lighting_cost_current',
 'lighting_cost_potential',
 'l

## 2. Completeness: missing values

In [25]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)
missing_pct.head(20)

sheating_env_eff            100.0
sheating_energy_eff         100.0
floor_env_eff                96.4
floor_energy_eff             96.4
secondheat_description       91.7
address3                     82.8
roof_env_eff                 63.2
roof_energy_eff              63.2
unheated_corridor_length     57.0
heat_loss_corridor           35.1
glazed_area                  34.9
glazed_type                  34.9
photo_supply                 28.5
mains_gas_flag               28.4
address2                     23.6
mechanical_ventilation       11.6
number_heated_rooms          11.6
solar_water_heating_flag     11.6
number_habitable_rooms       11.6
extension_count              11.6
dtype: float64

In [26]:
critical = ["uprn", "certificate_number", "postcode", "lodgement_date",
            "total_floor_area", "current_energy_rating", "construction_age_band", "tenure"]
missing_pct[critical].sort_values(ascending=False)

uprn                     9.1
tenure                   4.7
postcode                 0.0
certificate_number       0.0
lodgement_date           0.0
total_floor_area         0.0
current_energy_rating    0.0
construction_age_band    0.0
dtype: float64

### Findings: completeness

The core certificate fields are in good health: postcode, certificate_number,
lodgement_date, total_floor_area and current_energy_rating are all effectively
~100% complete across 102,185 records.

The gaps are concentrated where they hurt most:

- **9.1% of certificates (~9,300) have no UPRN**, the unique property identifier.
  These records cannot be reliably linked to a specific home, which undermines
  deduplication, property history tracking, and any address-level analysis.
- **11.6% of records (~11,900 homes) are missing basic property facts** such as
  number_habitable_rooms and number_heated_rooms. Five fields share this exact
  rate, suggesting a structural cause (e.g. certain assessment types omitting
  them) rather than random entry errors: to be investigated.
- **4.7% lack tenure** (owner-occupied vs rented), a gap with regulatory weight
  since minimum energy standards for rentals depend on knowing tenure.
- Two columns (sheating_env_eff, sheating_energy_eff) are 100% empty: dead
  columns carried in the extract.
- High missingness in address2 (23.6%) and address3 (82.8%) **is expected**, not a
  defect: most properties simply need fewer address lines.

## 3. Uniqueness: duplicate records

In [27]:
exact_dupes = df.duplicated().sum()
cert_dupes = df["certificate_number"].duplicated().sum()
print(f"Exact duplicate rows: {exact_dupes}")
print(f"Duplicate certificate numbers: {cert_dupes}")

Exact duplicate rows: 0
Duplicate certificate numbers: 0


In [28]:
has_uprn = df[df["uprn"].notna()]
props = has_uprn["uprn"].nunique()
certs = len(has_uprn)
multi = has_uprn["uprn"].value_counts()
multi_props = (multi > 1).sum()
print(f"Certificates with a UPRN: {certs}")
print(f"Unique properties (UPRNs): {props}")
print(f"Properties with more than one certificate: {multi_props}")
print(f"Max certificates for a single property: {multi.max()}")

Certificates with a UPRN: 92883
Unique properties (UPRNs): 71513
Properties with more than one certificate: 17405
Max certificates for a single property: 69


In [29]:
top_uprn = multi.idxmax()
champion = has_uprn[has_uprn["uprn"] == top_uprn]
champion[["certificate_number", "address", "postcode", "lodgement_date",
          "current_energy_rating", "total_floor_area", "transaction_type"]].sort_values("lodgement_date")

,certificate_number,address,postcode,lodgement_date,current_energy_rating,total_floor_area,transaction_type
49539,0340-3442-0060-2707-2581,"E.3.1, 4 Lewis Cubitt Walk",N1C 4DY,2023-06-02,B,81,New dwelling
40163,7137-0736-4000-0472-5202,"E.4.1, 4 Lewis Cubitt Walk",N1C 4DY,2023-06-02,B,81,New dwelling
36414,0380-3162-3060-2407-6581,"E.11.3, 4 Lewis Cubitt Walk",N1C 4DY,2023-06-02,B,39,New dwelling
35398,0226-3005-1306-4847-7204,"E.11.1, 4 Lewis Cubitt Walk",N1C 4DY,2023-06-02,B,81,New dwelling
49366,0629-3005-4306-1387-4200,"E.11.5, 4 Lewis Cubitt Walk",N1C 4DY,2023-06-02,B,57,New dwelling
...,...,...,...,...,...,...,...
65418,3837-2430-1009-0877-5206,"E.13.4, 4 Lewis Cubitt Walk",N1C 4DY,2023-10-03,B,52,New dwelling
63253,9310-3787-6000-2407-2545,"E.13.5, 4 Lewis Cubitt Walk",N1C 4DY,2023-10-03,B,57,New dwelling
44666,6837-2430-1009-0807-5202,"E.14.4, 4 Lewis Cubitt Walk",N1C 4DY,2023-10-03,B,53,New dwelling
10324,0374-3905-3300-8367-4204,"E.13.3, 4 Lewis Cubitt Walk",N1C 4DY,2023-10-03,B,39,New dwelling


In [30]:
top10 = multi.head(15)
for uprn, count in top10.items():
    rows = has_uprn[has_uprn["uprn"] == uprn]
    print(f"UPRN {uprn}: {count} certs, {rows['address'].nunique()} distinct addresses, "
          f"postcode {rows['postcode'].iloc[0]}")

UPRN 5200862.0: 69 certs, 69 distinct addresses, postcode N1C 4DY
UPRN 5083282.0: 45 certs, 45 distinct addresses, postcode EC1R 5EG
UPRN 5082824.0: 32 certs, 30 distinct addresses, postcode WC1N 1DD
UPRN 5067987.0: 31 certs, 25 distinct addresses, postcode NW1 0DR
UPRN 5097188.0: 29 certs, 17 distinct addresses, postcode NW3 5JY
UPRN 5129025.0: 29 certs, 29 distinct addresses, postcode NW1 1DT
UPRN 5169012.0: 28 certs, 28 distinct addresses, postcode EC1N 8TE
UPRN 5169011.0: 28 certs, 28 distinct addresses, postcode EC1N 8TE
UPRN 5045735.0: 28 certs, 25 distinct addresses, postcode NW1 0DJ
UPRN 5169010.0: 28 certs, 28 distinct addresses, postcode EC1N 8TE
UPRN 5036027.0: 26 certs, 20 distinct addresses, postcode NW5 2EH
UPRN 5169013.0: 25 certs, 24 distinct addresses, postcode EC1N 8TE
UPRN 5169747.0: 22 certs, 22 distinct addresses, postcode EC1N 8TE
UPRN 5133088.0: 20 certs, 20 distinct addresses, postcode NW1 9BL
UPRN 5157614.0: 19 certs, 15 distinct addresses, postcode NW6 4LJ


In [31]:
suspect = has_uprn[has_uprn["uprn"] == 5097188.0]
suspect[["address", "lodgement_date", "current_energy_rating",
         "total_floor_area", "transaction_type"]].sort_values(["address", "lodgement_date"])

,address,lodgement_date,current_energy_rating,total_floor_area,transaction_type
74422,"Flat 1, 11, Fitzjohns Avenue",2016-01-28,D,207,Marketed sale
70263,"Flat 1/A, 11, Fitzjohns Avenue",2016-01-28,D,23,Marketed sale
27389,"Flat 1/A, 11, Fitzjohns Avenue",2018-11-16,C,119,New dwelling
19450,"Flat 10/A, 11, Fitzjohns Avenue",2016-01-28,D,27,Marketed sale
91117,"Flat 10/A, 11, Fitzjohns Avenue",2018-11-16,C,49,New dwelling
83859,"Flat 11, 21, Fitzjohns Avenue",2018-07-16,E,150,Marketed sale
60871,"Flat 11/A, 11, Fitzjohns Avenue",2016-01-28,D,30,Marketed sale
85261,"Flat 11/A, 11, Fitzjohns Avenue",2018-11-16,C,70,New dwelling
27190,"Flat 12/A, 11, Fitzjohns Avenue",2016-01-28,E,23,Marketed sale
55057,"Flat 12/A, 11, Fitzjohns Avenue",2018-11-16,C,71,New dwelling


In [32]:
df["transaction_type"].value_counts(dropna=False)

transaction_type
Rental                       53680
Marketed sale                25177
New dwelling                 10572
None of the above             3474
Non-marketed sale             1761
marketed sale                 1408
rental (private)              1399
Assessment for Green Deal     1378
Stock condition survey         946
ECO assessment                 942
rental (social)                454
new dwelling                   388
NaN                            268
FiT application                 91
Grant scheme                    64
Following Green Deal            54
non marketed sale               53
RHI application                 32
not sale or rental              19
Re-mortgaging                   14
Non-grant scheme                11
Name: count, dtype: int64

In [33]:
suspect[["address", "lodgement_date", "transaction_type", "report_type",
         "inspection_date", "construction_age_band"]].sort_values(["address", "lodgement_date"])

,address,lodgement_date,transaction_type,report_type,inspection_date,construction_age_band
74422,"Flat 1, 11, Fitzjohns Avenue",2016-01-28,Marketed sale,2,2016-01-26,England and Wales: before 1900
70263,"Flat 1/A, 11, Fitzjohns Avenue",2016-01-28,Marketed sale,2,2016-01-26,England and Wales: before 1900
27389,"Flat 1/A, 11, Fitzjohns Avenue",2018-11-16,New dwelling,3,2018-11-16,2017
19450,"Flat 10/A, 11, Fitzjohns Avenue",2016-01-28,Marketed sale,2,2016-01-26,England and Wales: before 1900
91117,"Flat 10/A, 11, Fitzjohns Avenue",2018-11-16,New dwelling,3,2018-11-16,2017
83859,"Flat 11, 21, Fitzjohns Avenue",2018-07-16,Marketed sale,2,2018-07-16,England and Wales: 1900-1929
60871,"Flat 11/A, 11, Fitzjohns Avenue",2016-01-28,Marketed sale,2,2016-01-26,England and Wales: before 1900
85261,"Flat 11/A, 11, Fitzjohns Avenue",2018-11-16,New dwelling,3,2018-11-16,2017
27190,"Flat 12/A, 11, Fitzjohns Avenue",2016-01-28,Marketed sale,2,2016-01-26,England and Wales: before 1900
55057,"Flat 12/A, 11, Fitzjohns Avenue",2018-11-16,New dwelling,3,2018-11-16,2017


In [34]:
same_day = has_uprn.groupby(["uprn", "address", "lodgement_date"]).size()
true_dupes = same_day[same_day > 1]
print(f"Same property, same address, same day, multiple certificates: {true_dupes.sum() - len(true_dupes)} extra certificates")
print(f"Affected property-days: {len(true_dupes)}")
true_dupes.sort_values(ascending=False).head(20)

Same property, same address, same day, multiple certificates: 475 extra certificates
Affected property-days: 452


uprn       address                                         lodgement_date
5059917.0  Flat 33 Greenwood, Oseney Crescent              2015-05-12        5
5033492.0  Flat 38, Bacton, Haverstock Road                2025-01-08        4
5182918.0  Apartment 203 Orwell Building, Heritage Lane    2017-01-13        4
5198791.0  Flat 4, 22 Theobald Road                        2021-03-30        4
5202256.0  7 Milburn Lane, Agar Grove                      2024-03-28        4
5007575.0  Flat 55 Henderson Court, 102, Fitzjohns Avenue  2015-02-24        3
5019361.0  Flat 7/A Rose Bush Court, 35, Parkhill Road     2012-05-03        3
5195683.0  Flat 2, 25 Montpelier Grove                     2013-03-26        3
5055943.0  3, Agar Grove                                   2014-11-20        3
5059347.0  Flat 205 Denton, Malden Crescent                2013-10-17        3
5066532.0  42 Mornington Crescent, Camden                  2023-04-19        3
5080949.0  6a, Mortimer Crescent                         

In [35]:
flat33 = has_uprn[(has_uprn["uprn"] == 5059917.0) & (has_uprn["lodgement_date"] == "2015-05-12")]
flat33[["certificate_number", "address", "current_energy_rating",
        "current_energy_efficiency", "total_floor_area", "co2_emissions_current",
        "inspection_date", "lodgement_datetime"]]

,certificate_number,address,current_energy_rating,current_energy_efficiency,total_floor_area,co2_emissions_current,inspection_date,lodgement_datetime
8932,8935-7024-3200-0523-0906,"Flat 33 Greenwood, Oseney Crescent",D,56,36,2.8,2015-04-07,2015-05-12 19:34:49
29736,8635-7024-3200-0553-0906,"Flat 33 Greenwood, Oseney Crescent",D,56,36,2.8,2015-04-07,2015-05-12 08:58:05
46325,0330-2853-7240-9005-2015,"Flat 33 Greenwood, Oseney Crescent",D,56,36,2.8,2015-04-07,2015-05-12 19:36:11
61661,0438-3020-7204-3555-0904,"Flat 33 Greenwood, Oseney Crescent",D,56,36,2.8,2015-04-07,2015-05-12 08:54:22
73860,8535-7024-3200-0523-0902,"Flat 33 Greenwood, Oseney Crescent",D,56,36,2.8,2015-04-07,2015-05-12 09:11:10


### Findings: uniqueness

The good news: across all 102,185 records there are **zero duplicate rows and
zero repeated certificate numbers**. As a list of certificates, the data is
clean.

The problems appear when you ask about properties instead of certificates:

- **Certificate counts are not property counts.** The 92,883 certificates with
  a UPRN cover only **71,513 distinct properties**. About 1 in 4 certificates
  is a repeat visit to a home already in the data, and **17,405 properties have
  more than one certificate**. Getting a new certificate is normal (homes are
  reassessed when sold or re-let), but anyone counting certificates as homes
  overstates Camden's housing stock by over 20%.

- **One identifier can hide an entire building.** The most repeated UPRN in
  Camden covers **69 different flats** in a new King's Cross development, and
  most of the top 15 repeated UPRNs are whole apartment blocks sharing one ID.
  The UPRN is supposed to answer "which home is this?", but here it only
  answers "which building?". This means our 71,513 figure undercounts the
  real number of homes, and deduplicating by UPRN would wrongly merge dozens
  of separate flats into one.

- **The same address can mean two different homes.** At one Hampstead building,
  the same flat addresses appear twice with contradicting facts: a flat is
  23 m2, rated D, in a "before 1900" building in 2016, then 119 m2, rated C,
  built "2017" in 2018. The building was sold and rebuilt; the old addresses
  now point at physically different flats, and several 2016 flats have no
  newer record and may no longer exist.

- **Nothing in the data flags any of this.** There is no status field, no link
  between old and new certificates, no "superseded" marker. The rebuild above
  is only detectable by cross-referencing columns and dates by hand. Stale
  records for vanished flats look exactly like current ones, so any "current
  state" analysis has to work around this, usually by keeping only the latest
  certificate per property.
- **True duplicate lodgements exist.** 475 surplus certificates were found where
  the same property, at the same address, was certificated more than once on
  the same day (452 property-days affected). In the worst case, one Kentish
  Town flat received **five certificates in a single day**, all substantively
  identical (same rating, score, floor area and emissions, lodged minutes
  apart), one assessment registered five times. Certificate numbers are unique,
  but nothing prevents the same assessment being lodged repeatedly.
  
Note: the 9.1% of certificates without a UPRN are excluded from these checks.
Duplicates among them are checked separately using address matching below.